# SignEdge · prática

Rode as células em ordem, junto com o professor. Onde houver `# experimente`, mude o valor e rode de novo.

## Preparação

Rode no intervalo (~2 min). Se pedir um arquivo, escolha o `minicurso.zip` do pendrive.

<sub>Fotos da base LIBRAS-HC-RGBDS (UFPR): uso acadêmico livre, uso comercial não permitido.</sub>

In [ ]:
#@title Preparação
import os, sys, time, zipfile, subprocess, urllib.request, warnings
import importlib.metadata as md
from pathlib import Path

KIT_URL = "https://github.com/G1L404/SignEdge/releases/download/minicurso-ercemapi-2026/minicurso.zip"
t0 = time.time()
EM_COLAB = "google.colab" in sys.modules
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")
warnings.filterwarnings("ignore")

RAIZ = Path.cwd() / "signedge"
if not (RAIZ / "kit/apoio.py").exists():
    zp = Path("minicurso.zip")
    if zp.exists() and not zipfile.is_zipfile(zp):
        zp.unlink()
    if not zp.exists():
        try:
            print("baixando o kit...")
            urllib.request.urlretrieve(KIT_URL, zp)
            if not zipfile.is_zipfile(zp):
                zp.unlink()
                raise ValueError("o link não entregou um arquivo .zip")
        except Exception as erro:
            print("não consegui baixar:", erro)
            if not EM_COLAB:
                raise
            print("escolha o minicurso.zip do pendrive:")
            from google.colab import files
            files.upload()
    zipfile.ZipFile(zp).extractall(RAIZ)
KIT = RAIZ / "kit"

def versao(pacote):
    try:
        return md.version(pacote)
    except md.PackageNotFoundError:
        return None

if versao("litert-torch") is None:
    if not (EM_COLAB or os.environ.get("SIGNEDGE_INSTALAR") == "1"):
        raise SystemExit("falta o litert-torch. Use o Colab, ou um ambiente separado com SIGNEDGE_INSTALAR=1")
    print("instalando as bibliotecas...")
    pip = [sys.executable, "-m", "pip", "install", "-q"]
    tv = versao("torch")
    if tv is None or tuple(int(x) for x in tv.split("+")[0].split(".")[:2]) >= (2, 14):
        # o conversor para TFLite pede torch < 2.14
        subprocess.run(pip + ["torch==2.13.*", "torchvision==0.28.*", "--index-url", "https://download.pytorch.org/whl/cpu"], check=True)
    subprocess.run(pip + ["litert-torch==0.9.4", "timm==1.0.30", "pyyaml"], check=True)

os.chdir(RAIZ)
sys.path[:0] = [str(RAIZ), str(KIT)]
print(f"pronto em {time.time() - t0:.0f} s")

## 0. Ponto de partida

In [ ]:
import copy
import numpy as np
import torch
import apoio as ap

aluno, aluno_ce = ap.iniciar(KIT)
notas = ap.respostas(aluno)
ap.tabela([("aluno", {**ap.acerto(notas), "MiB": ap.mb(aluno)})])

## 1. Destilação

In [ ]:
ap.professor()

In [ ]:
ap.professor(cm=46)   # experimente outra CM, de 1 a 61

In [ ]:
ap.professor(temperatura=2)   # experimente 1, 2, 4

In [ ]:
ap.misturas()

In [ ]:
import torch.nn.functional as F

def perda_destilacao(notas_aluno, notas_professor, T=1.0):
    """Divergência KL entre a resposta do professor e a do aluno, com temperatura T."""
    log_p_aluno = F.log_softmax(notas_aluno / T, dim=1)
    p_professor = F.softmax(notas_professor / T, dim=1)
    return F.kl_div(log_p_aluno, p_professor, reduction="batchmean") * T * T

# no treino: perda = perda_destilacao(aluno(x), professor(x))
notas_prof = ap.D["prof_logits"].astype(np.float32)
notas_sem_professor = ap.respostas(aluno_ce)
for nome, n in (("aluno sem professor", notas_sem_professor), ("aluno com professor", notas)):
    print(f"{nome:20}: distância até o professor = {perda_destilacao(torch.tensor(n), torch.tensor(notas_prof)):.2f}")

In [ ]:
ap.tabela([
    ("professor (28 M de pesos)", {**ap.acerto(notas_prof), "MiB": 106.3}),
    ("aluno sem professor", {**ap.acerto(notas_sem_professor), "MiB": ap.mb(aluno_ce)}),
    ("aluno com professor", {**ap.acerto(notas), "MiB": ap.mb(aluno)}),
])

## 2. Poda

In [ ]:
from research.optimization.pruner import attach_masks, bake, compute_masks, prunable, set_masks, sparsity_report

def podar(modelo, criterio, fracao):
    """Zera `fracao` dos pesos das convoluções 1×1, escolhidos pelo `criterio`."""
    m = copy.deepcopy(modelo)
    attach_masks(m)
    camadas = [c for _, c in prunable(m)]
    set_masks(m, compute_masks(camadas, fracao, criterio))
    return bake(m)

l1 = podar(aluno, "l1_uniform", 0.5)
lamp = podar(aluno, "lamp", 0.5)
notas_l1, notas_lamp = ap.respostas(l1), ap.respostas(lamp)

ap.tabela([("inteiro", {**ap.acerto(notas), "zeros": ap.zeros(aluno)}),
           ("L1 uniforme 50%", {**ap.acerto(notas_l1), "zeros": ap.zeros(l1)}),
           ("LAMP 50%", {**ap.acerto(notas_lamp), "zeros": ap.zeros(lamp)})])
ap.grade(("inteiro", notas), ("L1 50%", notas_l1), ("LAMP 50%", notas_lamp))

In [ ]:
ap.camadas(sparsity_report(lamp))

In [ ]:
CRITERIO = "lamp"   # experimente "l1_uniform"
FRACAO = 0.6        # experimente 0.55, 0.65, 0.7
ap.tabela([(f"{CRITERIO} {FRACAO:.0%}", ap.acerto(ap.respostas(podar(aluno, CRITERIO, FRACAO))))])

### Recalibrar a confiança

In [ ]:
from research.optimization.calibration import fit_temperature

outras_pessoas = ap.D["oof_fold"] != ap.D["registro"]["fold"]
T = fit_temperature(ap.D["oof_poda_logits"][outras_pessoas], ap.D["oof_labels"][outras_pessoas])
print(f"temperatura ajustada: T = {T:.2f}\n")
ap.tabela([("LAMP 50%, sem temperatura", ap.acerto(notas_lamp)),
           (f"LAMP 50%, com T = {T:.2f}", ap.acerto(notas_lamp, T))])
ap.confianca(("inteiro", notas, 1.0), ("LAMP 50%\nsem temperatura", notas_lamp, 1.0),
             (f"LAMP 50%\ncom T = {T:.2f}", notas_lamp, T))

In [ ]:
T_TESTE = 0.76   # experimente 1.0, 0.3
ap.tabela([(f"LAMP 50%, T = {T_TESTE}", ap.acerto(notas_lamp, T_TESTE))])

## 3. Quantização

In [ ]:
ap.regua(aluno)

In [ ]:
# leva cerca de 1,5 min no Colab
from research.optimization.quantizer import export_float, quantize_int8, scale_head

SAIDA = Path("modelos")
calibracao = ap.calibracao_int8()

def converter(modelo, nome, T=1.0):
    m = scale_head(copy.deepcopy(modelo), T)   # 1/T embutido na última camada
    with ap.silencio():
        f32 = export_float(m, SAIDA / f"{nome}_float32.tflite", 224)
        i8 = quantize_int8(f32, calibracao, SAIDA / f"{nome}_int8.tflite", landmarks_float=True)
    return f32, i8

t0 = time.time()
inteiro_f32, inteiro_i8 = converter(aluno, "inteiro")
podado_f32, podado_i8 = converter(lamp, "lamp50", T)
print(f"4 arquivos em {time.time() - t0:.0f} s")

In [ ]:
ap.arquivos(inteiro_f32, inteiro_i8, podado_f32, podado_i8)

## 4. Implantação

In [ ]:
# no seu Android: veja a folha do aluno
PASTA_APP = "/storage/emulated/0/Android/data/br.ufpr.signedge.signedge_mobile/files"
if EM_COLAB:
    from google.colab import files
    files.download(str(podado_i8))
print(f"arquivo: {podado_i8.name} ({podado_i8.stat().st_size / 2**20:.2f} MiB)\n")
print("no terminal do computador, na pasta do arquivo baixado:")
print(f"  adb push {podado_i8.name} {PASTA_APP}/model_int8.tflite")
print("  adb shell am force-stop br.ufpr.signedge.signedge_mobile")

In [ ]:
ap.testar_cm(podado_i8, cm=38)   # experimente outras CMs

## Leve para casa

In [ ]:
import torchvision, litert_torch
from ai_edge_quantizer import quantizer, recipe

minha_cnn = torchvision.models.mobilenet_v3_small(num_classes=10).eval()   # experimente resnet18, efficientnet_b0

# poda
podada = podar(minha_cnn, "lamp", 0.5)

# temperatura: T = fit_temperature(notas_de_validacao, rotulos), depois scale_head(modelo, T)

with ap.silencio():
    # TFLite
    litert_torch.convert(podada, (torch.rand(1, 3, 224, 224),)).export("minha_cnn_float32.tflite")
    # Int8 só nos pesos
    q = quantizer.Quantizer("minha_cnn_float32.tflite", recipe.dynamic_wi8_afp32())
    q.quantize().export_model("minha_cnn_int8.tflite")

for f in ("minha_cnn_float32.tflite", "minha_cnn_int8.tflite"):
    print(f"{f:26} {os.path.getsize(f) / 2**20:.2f} MiB")